# Graph Similarity Analysis Notebook

This notebook compares:

1. A **semantic embeddings graph matrix** (distance or similarity), and
2. A **subscriptions graph matrix** (distance or similarity).

It produces:

- **Numerical outputs** (correlation/error/overlap/permutation-test metrics), and
- **Visual outputs** (heatmaps + difference map + scatter).

The approach combines multiple perspectives, since no single graph metric captures all structural dimensions.

## Scientific grounding

This notebook operationalizes methods inspired by:

- Mantel, N. (1967), *Cancer Research* — matrix correlation with permutation testing.
- Krackhardt, D. (1987), *Social Networks* — QAP-style permutation logic for network association.
- Borgatti, Everett, & Johnson (2018), *Analyzing Social Networks* — interpretation of dyadic matrix comparisons.
- Schieber et al. (2017), *Nature Communications* — motivation for multi-metric network dissimilarity assessment.

In [ ]:
# Core scientific stack for matrix math, statistics, and plotting.
import json
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.spatial.distance import cosine
from scipy.stats import pearsonr, spearmanr


In [ ]:
# ----------------------------
# Reusable analysis primitives
# ----------------------------

@dataclass
class SimilarityResult:
    """Container for all numeric outputs from the comparison pipeline."""

    n_nodes: int
    n_pairs: int
    pearson_r: float
    pearson_p: float
    spearman_rho: float
    spearman_p: float
    cosine_similarity: float
    rmse: float
    mae: float
    edge_jaccard: float
    mantel_r: float
    mantel_p: float
    qap_r: float
    qap_p: float


def resolve_latest_drive_csv(source: Path, download_dir: Path) -> Path:
    """Copy the latest Google Drive CSV artifact to local runtime storage.

    `source` can be either:
    - a direct CSV file path,
    - a directory (the notebook will read `<dir>/adjacency_matrix.csv`), or
    - a glob-style pattern so the latest modified match is selected.
    """
    source = Path(str(source))
    source_str = str(source)

    if source.exists() and source.is_dir():
        candidate = source / 'adjacency_matrix.csv'
        if not candidate.exists():
            raise FileNotFoundError(f"Expected adjacency_matrix.csv under: {source}")
        source = candidate
        source_str = str(source)

    if any(ch in source_str for ch in "*?[]"):
        matches = sorted(Path('/').glob(source_str.lstrip('/')), key=lambda p: p.stat().st_mtime)
        if not matches:
            raise FileNotFoundError(f"No Google Drive CSV matched pattern: {source_str}")
        latest_source = matches[-1]
    else:
        if not source.exists():
            raise FileNotFoundError(f"Google Drive file not found: {source}")
        latest_source = source

    download_dir.mkdir(parents=True, exist_ok=True)
    destination = download_dir / latest_source.name
    shutil.copy2(latest_source, destination)
    print(f"Downloaded latest Drive artifact: {latest_source} -> {destination}")
    return destination


def load_square_matrix(csv_path: Path) -> pd.DataFrame:
    """Load a labeled square matrix from CSV and enforce validity checks.

    The CSV is expected to have a row index in the first column and matching
    row/column labels describing the same node set.
    """
    df = pd.read_csv(csv_path, index_col=0)

    if df.shape[0] != df.shape[1]:
        if df.shape[1] <= 3:
            raise ValueError(
                f"Matrix at {csv_path} is not square: {df.shape}. "
                "This looks like an edge-list export (deprecated). "
                "Use Graphiko schema adjacency matrix paths, e.g. "
                "/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv"
            )
        raise ValueError(f"Matrix at {csv_path} is not square: {df.shape}")

    df.index = df.index.map(str)
    df.columns = df.columns.map(str)

    row_labels = set(df.index)
    col_labels = set(df.columns)
    if row_labels != col_labels:
        raise ValueError(
            f"Row/column label mismatch in {csv_path}. "
            f"Rows={len(row_labels)} unique, Cols={len(col_labels)} unique"
        )

    # Guarantee row/column alignment by ID before any downstream analysis.
    df = df.reindex(index=sorted(df.index), columns=sorted(df.columns))
    df = df.loc[df.index, df.index]
    df = df.apply(pd.to_numeric, errors="raise")
    return df


def align_matrices(a: pd.DataFrame, b: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Restrict both matrices to the shared node intersection in a stable order."""
    common = sorted(set(a.index).intersection(b.index))
    if len(common) < 3:
        raise ValueError("Need at least 3 shared nodes for robust graph similarity stats.")
    return a.loc[common, common], b.loc[common, common]


def minmax_normalize(df: pd.DataFrame) -> pd.DataFrame:
    """Global min-max normalization to [0, 1] for cross-pipeline comparability."""
    arr = df.to_numpy(dtype=float)
    v_min, v_max = np.nanmin(arr), np.nanmax(arr)
    if np.isclose(v_min, v_max):
        return pd.DataFrame(np.zeros_like(arr), index=df.index, columns=df.columns)
    return pd.DataFrame((arr - v_min) / (v_max - v_min), index=df.index, columns=df.columns)


def distances_to_similarity(df: pd.DataFrame) -> pd.DataFrame:
    """Convert distances to similarities using similarity = 1 - normalized_distance."""
    d = minmax_normalize(df)
    return 1.0 - d


def upper_triangle_vector(df: pd.DataFrame, include_diagonal: bool = False) -> np.ndarray:
    """Vectorize matrix upper triangle, preserving undirected edge uniqueness."""
    arr = df.to_numpy(dtype=float)
    k = 0 if include_diagonal else 1
    idx = np.triu_indices_from(arr, k=k)
    return arr[idx]


def mantel_test(a: pd.DataFrame, b: pd.DataFrame, permutations: int = 2000, random_state: int = 42) -> Tuple[float, float]:
    """Mantel correlation with permutation p-value.

    Two-sided significance is estimated by permuting one matrix with node-label
    preserving row/column permutations.
    """
    rng = np.random.default_rng(random_state)
    a_vec, b_vec = upper_triangle_vector(a), upper_triangle_vector(b)
    obs_r, _ = pearsonr(a_vec, b_vec)

    n = a.shape[0]
    b_arr = b.to_numpy(dtype=float)
    perm_stats = np.empty(permutations, dtype=float)

    for i in range(permutations):
        p = rng.permutation(n)
        b_perm = b_arr[np.ix_(p, p)]
        perm_stats[i], _ = pearsonr(a_vec, upper_triangle_vector(pd.DataFrame(b_perm)))

    p_value = (np.sum(np.abs(perm_stats) >= np.abs(obs_r)) + 1) / (permutations + 1)
    return float(obs_r), float(p_value)


def qap_correlation(a: pd.DataFrame, b: pd.DataFrame, permutations: int = 2000, random_state: int = 7) -> Tuple[float, float]:
    """QAP-style matrix correlation with one-sided permutation p-value."""
    rng = np.random.default_rng(random_state)
    a_vec, b_vec = upper_triangle_vector(a), upper_triangle_vector(b)
    obs_r, _ = pearsonr(a_vec, b_vec)

    n = a.shape[0]
    b_arr = b.to_numpy(dtype=float)
    perm_stats = np.empty(permutations, dtype=float)

    for i in range(permutations):
        p = rng.permutation(n)
        b_perm = b_arr[np.ix_(p, p)]
        perm_stats[i], _ = pearsonr(a_vec, upper_triangle_vector(pd.DataFrame(b_perm)))

    p_value = (np.sum(perm_stats >= obs_r) + 1) / (permutations + 1)
    return float(obs_r), float(p_value)


def compute_similarity_metrics(
    embeddings_matrix: pd.DataFrame,
    subscriptions_matrix: pd.DataFrame,
    edge_threshold: float,
    permutations: int,
) -> SimilarityResult:
    """Compute a suite of complementary graph similarity metrics."""
    emb_vec = upper_triangle_vector(embeddings_matrix)
    sub_vec = upper_triangle_vector(subscriptions_matrix)

    pear_r, pear_p = pearsonr(emb_vec, sub_vec)
    spr_rho, spr_p = spearmanr(emb_vec, sub_vec)
    cos_sim = 1.0 - cosine(emb_vec, sub_vec)

    rmse = float(np.sqrt(np.mean((emb_vec - sub_vec) ** 2)))
    mae = float(np.mean(np.abs(emb_vec - sub_vec)))

    emb_edge = emb_vec >= edge_threshold
    sub_edge = sub_vec >= edge_threshold
    inter = np.logical_and(emb_edge, sub_edge).sum()
    union = np.logical_or(emb_edge, sub_edge).sum()
    edge_jaccard = float(inter / union) if union > 0 else float("nan")

    mantel_r, mantel_p = mantel_test(embeddings_matrix, subscriptions_matrix, permutations)
    qap_r, qap_p = qap_correlation(embeddings_matrix, subscriptions_matrix, permutations)

    return SimilarityResult(
        n_nodes=embeddings_matrix.shape[0],
        n_pairs=emb_vec.size,
        pearson_r=float(pear_r),
        pearson_p=float(pear_p),
        spearman_rho=float(spr_rho),
        spearman_p=float(spr_p),
        cosine_similarity=float(cos_sim),
        rmse=rmse,
        mae=mae,
        edge_jaccard=edge_jaccard,
        mantel_r=mantel_r,
        mantel_p=mantel_p,
        qap_r=qap_r,
        qap_p=qap_p,
    )


def save_visualizations(embeddings_matrix: pd.DataFrame, subscriptions_matrix: pd.DataFrame, output_dir: Path) -> Dict[str, str]:
    """Render and save graph comparison figures."""
    output_dir.mkdir(parents=True, exist_ok=True)
    out: Dict[str, str] = {}

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
    sns.heatmap(embeddings_matrix, cmap="viridis", ax=axes[0], cbar=True)
    axes[0].set_title("Embeddings Graph (normalized)")
    sns.heatmap(subscriptions_matrix, cmap="viridis", ax=axes[1], cbar=True)
    axes[1].set_title("Subscriptions Graph (normalized)")
    p1 = output_dir / "heatmaps_side_by_side.png"
    fig.savefig(p1, dpi=220)
    plt.close(fig)
    out["heatmaps"] = str(p1)

    diff = embeddings_matrix - subscriptions_matrix
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.heatmap(diff, cmap="coolwarm", center=0.0, ax=ax)
    ax.set_title("Difference Matrix (Embeddings - Subscriptions)")
    p2 = output_dir / "difference_heatmap.png"
    fig.savefig(p2, dpi=220)
    plt.close(fig)
    out["difference_heatmap"] = str(p2)

    emb_vec = upper_triangle_vector(embeddings_matrix)
    sub_vec = upper_triangle_vector(subscriptions_matrix)
    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    sns.regplot(x=emb_vec, y=sub_vec, scatter_kws={"alpha": 0.45, "s": 18}, ax=ax)
    ax.set_xlabel("Embeddings edge weight")
    ax.set_ylabel("Subscriptions edge weight")
    ax.set_title("Edge-wise relationship between graphs")
    p3 = output_dir / "edge_weight_scatter.png"
    fig.savefig(p3, dpi=220)
    plt.close(fig)
    out["scatter"] = str(p3)

    return out


## Configure your run

Update the two Google Drive input paths (or glob patterns) below.

- The notebook copies the **latest Google Drive artifact** for each input into local runtime storage before analysis.
- If your files represent **distance matrices**, keep `DISTANCE_TO_SIMILARITY = True`.
- If they are already **similarities**, set it to `False`.


In [ ]:
# ----------------------------
# User configuration
# ----------------------------
EMBEDDINGS_DRIVE_SOURCE = Path('/content/drive/MyDrive/Graphiko/graphs/embeddings_distance/latest/adjacency_matrix.csv')
SUBSCRIPTIONS_DRIVE_SOURCE = Path('/content/drive/MyDrive/Graphiko/graphs/subscriptions_normalized_distance/latest/adjacency_matrix.csv')
LOCAL_DOWNLOAD_DIR = Path('inputs/downloaded_from_drive')
OUTPUT_DIR = Path('outputs/graph_similarity')

DISTANCE_TO_SIMILARITY = True
EDGE_THRESHOLD = 0.7
PERMUTATIONS = 2000


In [ ]:
# ----------------------------
# Execute analysis
# ----------------------------
EMBEDDINGS_CSV = resolve_latest_drive_csv(EMBEDDINGS_DRIVE_SOURCE, LOCAL_DOWNLOAD_DIR)
SUBSCRIPTIONS_CSV = resolve_latest_drive_csv(SUBSCRIPTIONS_DRIVE_SOURCE, LOCAL_DOWNLOAD_DIR)

emb_raw = load_square_matrix(EMBEDDINGS_CSV)
sub_raw = load_square_matrix(SUBSCRIPTIONS_CSV)

emb_aligned, sub_aligned = align_matrices(emb_raw, sub_raw)

if DISTANCE_TO_SIMILARITY:
    emb_norm = distances_to_similarity(emb_aligned)
    sub_norm = distances_to_similarity(sub_aligned)
else:
    emb_norm = minmax_normalize(emb_aligned)
    sub_norm = minmax_normalize(sub_aligned)

result = compute_similarity_metrics(
    emb_norm,
    sub_norm,
    edge_threshold=EDGE_THRESHOLD,
    permutations=PERMUTATIONS,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
metrics = asdict(result)

metrics_json = OUTPUT_DIR / 'similarity_metrics.json'
metrics_csv = OUTPUT_DIR / 'similarity_metrics.csv'
summary_json = OUTPUT_DIR / 'run_summary.json'

with metrics_json.open('w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

pd.DataFrame([metrics]).to_csv(metrics_csv, index=False)
plots = save_visualizations(emb_norm, sub_norm, OUTPUT_DIR)

summary = {
    'inputs': {
        'embeddings_drive_source': str(EMBEDDINGS_DRIVE_SOURCE),
        'subscriptions_drive_source': str(SUBSCRIPTIONS_DRIVE_SOURCE),
        'embeddings_downloaded_csv': str(EMBEDDINGS_CSV),
        'subscriptions_downloaded_csv': str(SUBSCRIPTIONS_CSV),
        'distance_to_similarity': DISTANCE_TO_SIMILARITY,
    },
    'metrics_json': str(metrics_json),
    'metrics_csv': str(metrics_csv),
    'plots': plots,
}

with summary_json.open('w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

print('Graph similarity analysis complete.')
print(json.dumps(summary, indent=2))
pd.DataFrame([metrics]).T.rename(columns={0: 'value'})
